# Exercise MegaDetector on Local Images

This notebook runs MegaDetector on a local image folder:

- `/Users/elhorte/Pictures/project-id-tests`

It is safe to run even when the folder is currently empty.

## Step 1 — Define paths

In [ ]:
from pathlib import Path

repo_root = Path.cwd().resolve()
md_root = repo_root / "third-party" / "MegaDetector"
image_dir = Path("/Volumes/BigMacX/ebio/Pictures/project-id")
output_dir = Path("/Volumes/BigMacX/ebio/project-id/outputs/megadetector")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {repo_root}")
print(f"MegaDetector path: {md_root}")
print(f"Image directory: {image_dir}")
print(f"Output directory: {output_dir}")

## Step 2 — Check image folder contents

In [ ]:
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
images = sorted([p for p in image_dir.glob("**/*") if p.is_file() and p.suffix.lower() in image_exts])

print(f"Found {len(images)} image(s)")
for p in images[:20]:
    print(" -", p)

if not images:
    print("\nNo images found yet. Populate the folder and rerun this notebook.")

## Step 3 — Build an image list file for MegaDetector batch inference

In [ ]:
image_list_file = output_dir / "image_list.txt"
image_list_file.write_text("\n".join(str(p) for p in images), encoding="utf-8")
print(f"Wrote: {image_list_file}")

## Step 4 — Run MegaDetector (batch mode)

This uses the MegaDetector batch detection script and writes a JSON results file.

> If your local MegaDetector checkout uses a different entry script, update the command in the next cell accordingly.

In [ ]:
import subprocess
import sys

results_json = output_dir / "megadetector_results.json"
batch_script = md_root / "megadetector" / "detection" / "run_detector_batch.py"

if not md_root.exists():
    raise FileNotFoundError(f"MegaDetector repo not found: {md_root}")
if not batch_script.exists():
    raise FileNotFoundError(f"Batch script not found: {batch_script}")
if len(images) == 0:
    raise RuntimeError("No images available for inference. Add images and rerun.")

cmd = [
    sys.executable,
    str(batch_script),
    "MDV5A",
    str(image_list_file),
    str(results_json),
    "--recursive",
]

print("Running command:")
print(" ".join(cmd))
subprocess.run(cmd, check=True)
print(f"\nDone. Results saved to: {results_json}")

## Step 5 — Preview detections summary

In [ ]:
import json
from collections import Counter

results_json = output_dir / "megadetector_results.json"
data = json.loads(results_json.read_text(encoding="utf-8"))
images_data = data.get("images", [])

print(f"Images in results: {len(images_data)}")

detections_per_image = Counter()
for item in images_data:
    detections_per_image[len(item.get("detections", []))] += 1

print("Detection count distribution (detections -> number of images):")
for k in sorted(detections_per_image):
    print(f"  {k} -> {detections_per_image[k]}")

## Step 6 — Visualize detections (bounding-box QA)

Draws MegaDetector bounding boxes on detected images and saves annotated copies to
`/Volumes/BigMacX/ebio/project-id/outputs/megadetector/annotated`.

Set `annotate_all = True` to annotate **every** successfully detected image; leave it
`False` to only process the first `max_preview` images. Regardless of the mode, up to
`max_preview` annotated images are shown inline so the notebook stays responsive.

In [ ]:
import json
from pathlib import Path

from IPython.display import Image as IPyImage, display
import megadetector.visualization.visualization_utils as vis_utils

# Reuse output_dir from Step 1; results file from Step 4.
results_json = output_dir / "megadetector_results.json"
results = json.loads(results_json.read_text(encoding="utf-8"))
category_map = results.get("detection_categories", {})

# --- Options ---------------------------------------------------------------
confidence_threshold = 0.2   # only draw boxes at/above this confidence
annotate_all = True          # True: annotate every detected image; False: only a preview
max_preview = 6              # how many annotated images to show inline (either mode)
# ---------------------------------------------------------------------------

# Saves to /Volumes/BigMacX/ebio/project-id/outputs/megadetector/annotated
annotated_dir = output_dir / "annotated"
annotated_dir.mkdir(parents=True, exist_ok=True)

images_with_detections = [
    im for im in results.get("images", [])
    if not im.get("failure")
    and any(d.get("conf", 0) >= confidence_threshold for d in im.get("detections", []))
]

total = len(images_with_detections)
print(f"{total} image(s) have detections at conf >= {confidence_threshold}")
if not images_with_detections:
    print("Nothing to visualize yet. Run Step 4 on a folder that contains animals/people/vehicles.")

to_process = images_with_detections if annotate_all else images_with_detections[:max_preview]
print(f"Annotating {len(to_process)} image(s) "
      f"({'all detected' if annotate_all else 'preview only'}); "
      f"showing up to {max_preview} inline.")

saved = 0
for idx, im in enumerate(to_process):
    src = Path(im["file"])
    if not src.exists():
        print(f"  (skipped, missing file) {src}")
        continue

    image = vis_utils.load_image(str(src))
    vis_utils.render_detection_bounding_boxes(
        im["detections"], image,
        label_map=category_map,
        confidence_threshold=confidence_threshold,
        thickness=4,
    )

    out_path = annotated_dir / f"{src.stem}_annotated.png"
    image.save(out_path)
    saved += 1

    n_boxes = sum(1 for d in im["detections"] if d.get("conf", 0) >= confidence_threshold)
    print(f"  [{idx + 1}/{len(to_process)}] {src.name}: {n_boxes} box(es) -> {out_path.name}")

    if idx < max_preview:
        display(IPyImage(filename=str(out_path), width=700))

print(f"\nSaved {saved} annotated image(s) to: {annotated_dir}")